# Notebook Training Pipeline

Notebook ini sengaja dibuat sebagai versi **simple untuk capstone/reviewer**:

Heartz MVP:

1. TensorFlow/Keras **Functional API**, bukan Sequential.
2. Input model adalah raw waveform 1D: `(16000,)`, 16 kHz, 1 detik.
3. Layer pertama setelah Input adalah custom `MelSpectrogramLayer` berbasis `tf.signal.stft`.
4. CNN 2D from scratch, tanpa pretrained model.
5. Output 20 class softmax.
6. Training pakai custom loop `tf.GradientTape`, bukan `model.fit()`.
7. Loss custom categorical crossentropy + label smoothing `0.1`.
8. Manual early stopping.
9. TensorBoard logging.
10. Export `.keras` dan SavedModel.

In [ ]:
# lebih rapih install dari requirements.txt di venv 

# !pip install tensorflow tensorboard numpy soundfile scikit-learn matplotlib

## 1. Import library dan setup path

In [2]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

# Kalau notebook dijalankan dari folder notebooks/, pindahkan working directory ke root machine-learning/.
cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)
    print("Changed working directory to:", Path.cwd())
else:
    print("Working directory:", Path.cwd())

TensorFlow: 2.21.0
Changed working directory to: d:\Hemkerr\Heartz\machine-learning


## 2. Konfigurasi Heartz

In [3]:
@dataclass(frozen=True)
class HeartzConfig:
    sample_rate: int = 16_000
    duration_seconds: float = 1.0
    num_samples: int = 16_000
    num_classes: int = 20

    frame_length: int = 400       # 25 ms at 16 kHz
    frame_step: int = 160         # 10 ms at 16 kHz
    fft_length: int = 512
    num_mel_bins: int = 64
    lower_edge_hertz: float = 80.0
    upper_edge_hertz: float = 7_600.0

    batch_size: int = 16
    epochs: int = 50
    learning_rate: float = 1e-3
    label_smoothing: float = 0.1
    early_stopping_patience: int = 8
    min_delta: float = 1e-4
    seed: int = 42

CONFIG = HeartzConfig()

CLASS_NAMES = [
    "A", "I", "U", "E", "O",
    "Ba", "Bi", "Bu", "Be", "Bo",
    "Pa", "Pi", "Pu", "Pe", "Po",
    "Ma", "Mi", "Mu", "Me", "Mo",
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

DATA_DIR = Path("data/clean_wav")
OUTPUT_DIR = Path("outputs")
LOG_DIR = OUTPUT_DIR / "logs" / time.strftime("%Y%m%d-%H%M%S")
KERAS_MODEL_PATH = OUTPUT_DIR / "heartz_model.keras"
SAVED_MODEL_DIR = OUTPUT_DIR / "saved_model"
CLASS_NAMES_PATH = OUTPUT_DIR / "class_names.json"
CONFIG_PATH = OUTPUT_DIR / "heartz_config.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG.seed)
np.random.seed(CONFIG.seed)
tf.random.set_seed(CONFIG.seed)

print("Expected data directory:", DATA_DIR.resolve())
print("Classes:", CLASS_NAMES)

Expected data directory: D:\Hemkerr\Heartz\machine-learning\data\clean_wav
Classes: ['A', 'I', 'U', 'E', 'O', 'Ba', 'Bi', 'Bu', 'Be', 'Bo', 'Pa', 'Pi', 'Pu', 'Pe', 'Po', 'Ma', 'Mi', 'Mu', 'Me', 'Mo']


## 3. Scan dataset
Audio ideal:

- `.wav`
- sample rate 16 kHz
- durasi 1 detik
- satu file berisi satu pelafalan suku kata

In [ ]:
def scan_dataset(data_dir: Path) -> Tuple[List[str], List[int]]:
    paths: List[str] = []
    labels: List[int] = []

    if not data_dir.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {data_dir.resolve()}\n"
        )

    missing_dirs = [name for name in CLASS_NAMES if not (data_dir / name).exists()]
    if missing_dirs:
        print("WARNING - folder class belum ada:", missing_dirs)

    for class_name in CLASS_NAMES:
        class_dir = data_dir / class_name
        if not class_dir.exists():
            continue
        for wav_path in sorted(class_dir.glob("*.wav")):
            paths.append(str(wav_path))
            labels.append(CLASS_TO_ID[class_name])

    if not paths:
        raise ValueError(
            f"Tidak ada file .wav di {data_dir.resolve()}\n"
            "Isi dataset dulu sebelum training."
        )

    return paths, labels

paths, labels = scan_dataset(DATA_DIR)
print("Total wav files:", len(paths))

# Tampilkan distribusi per class
counts = {name: 0 for name in CLASS_NAMES}
for label_id in labels:
    counts[CLASS_NAMES[label_id]] += 1
counts

Total wav files: 6096


{'A': 345,
 'I': 375,
 'U': 330,
 'E': 240,
 'O': 135,
 'Ba': 156,
 'Bi': 48,
 'Bu': 162,
 'Be': 147,
 'Bo': 156,
 'Pa': 492,
 'Pi': 339,
 'Pu': 369,
 'Pe': 360,
 'Po': 396,
 'Ma': 588,
 'Mi': 603,
 'Mu': 222,
 'Me': 165,
 'Mo': 468}

## 4. Split train/validation/test

In [5]:
def stratified_split(
    paths: List[str],
    labels: List[int],
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    seed: int = 42,
):
    rng = random.Random(seed)
    by_class: Dict[int, List[str]] = {i: [] for i in range(CONFIG.num_classes)}

    for path, label in zip(paths, labels):
        by_class[label].append(path)

    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    test_paths, test_labels = [], []

    for label, class_paths in by_class.items():
        rng.shuffle(class_paths)
        n = len(class_paths)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_part = class_paths[:n_train]
        val_part = class_paths[n_train:n_train + n_val]
        test_part = class_paths[n_train + n_val:]

        train_paths.extend(train_part)
        train_labels.extend([label] * len(train_part))
        val_paths.extend(val_part)
        val_labels.extend([label] * len(val_part))
        test_paths.extend(test_part)
        test_labels.extend([label] * len(test_part))

    def shuffle_pair(p, y):
        pairs = list(zip(p, y))
        rng.shuffle(pairs)
        if not pairs:
            return [], []
        p2, y2 = zip(*pairs)
        return list(p2), list(y2)

    return (*shuffle_pair(train_paths, train_labels),
            *shuffle_pair(val_paths, val_labels),
            *shuffle_pair(test_paths, test_labels))

train_paths, train_labels, val_paths, val_labels, test_paths, test_labels = stratified_split(
    paths, labels, seed=CONFIG.seed
)

print("Train:", len(train_paths))
print("Val  :", len(val_paths))
print("Test :", len(test_paths))

Train: 4870
Val  : 600
Test : 626


## 5. Loader audio `.wav` sebagai raw waveform

Loader ini memakai `tf.audio.decode_wav` untuk membaca waveform.

In [6]:
def load_wav_tensor(path: tf.Tensor, label: tf.Tensor):
    audio_binary = tf.io.read_file(path)
    audio, sample_rate = tf.audio.decode_wav(
        audio_binary,
        desired_channels=1,
        desired_samples=CONFIG.num_samples,
    )

    # Dataset seharusnya sudah 16 kHz dari tim Data Science.
    # Assertion ini sengaja dibuat supaya salah sample rate cepat ketahuan.
    tf.debugging.assert_equal(
        sample_rate,
        CONFIG.sample_rate,
        message="Sample rate harus 16000 Hz. Resample dataset dulu di pipeline Data Science.",
    )

    waveform = tf.squeeze(audio, axis=-1)  # (16000,)
    waveform = tf.cast(waveform, tf.float32)
    one_hot = tf.one_hot(label, depth=CONFIG.num_classes)
    return waveform, one_hot


def make_dataset(paths: List[str], labels: List[int], training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=max(len(paths), 1), seed=CONFIG.seed, reshuffle_each_iteration=True)
    ds = ds.map(load_wav_tensor, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(CONFIG.batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False) if val_paths else None
test_ds = make_dataset(test_paths, test_labels, training=False) if test_paths else None

for xb, yb in train_ds.take(1):
    print("Waveform batch:", xb.shape)
    print("Label batch:", yb.shape)

Waveform batch: (16, 16000)
Label batch: (16, 20)


## 6. Custom Layer: MelSpectrogramLayer

Ini layer pertama model setelah Input. Layer ini mengubah waveform 1D menjadi log Mel-Spectrogram 2D di dalam graph TensorFlow memakai `tf.signal.stft`.

In [7]:
@tf.keras.utils.register_keras_serializable(package="Heartz")
class MelSpectrogramLayer(tf.keras.layers.Layer):
    def __init__(
        self,
        sample_rate: int = 16_000,
        frame_length: int = 400,
        frame_step: int = 160,
        fft_length: int = 512,
        num_mel_bins: int = 64,
        lower_edge_hertz: float = 80.0,
        upper_edge_hertz: float = 7_600.0,
        eps: float = 1e-6,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.sample_rate = int(sample_rate)
        self.frame_length = int(frame_length)
        self.frame_step = int(frame_step)
        self.fft_length = int(fft_length)
        self.num_mel_bins = int(num_mel_bins)
        self.lower_edge_hertz = float(lower_edge_hertz)
        self.upper_edge_hertz = float(upper_edge_hertz)
        self.eps = float(eps)
        self._mel_weight_matrix = None

    def build(self, input_shape):
        num_spectrogram_bins = self.fft_length // 2 + 1
        mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins=self.num_mel_bins,
            num_spectrogram_bins=num_spectrogram_bins,
            sample_rate=self.sample_rate,
            lower_edge_hertz=self.lower_edge_hertz,
            upper_edge_hertz=self.upper_edge_hertz,
        )
        self._mel_weight_matrix = tf.constant(mel_weight_matrix, dtype=tf.float32)
        super().build(input_shape)

    def call(self, inputs, training=None):
        waveform = tf.cast(inputs, tf.float32)
        if waveform.shape.rank == 3 and waveform.shape[-1] == 1:
            waveform = tf.squeeze(waveform, axis=-1)

        stft = tf.signal.stft(
            waveform,
            frame_length=self.frame_length,
            frame_step=self.frame_step,
            fft_length=self.fft_length,
            window_fn=tf.signal.hann_window,
            pad_end=True,
        )
        magnitude = tf.abs(stft)
        power_spectrogram = tf.square(magnitude)

        mel_spectrogram = tf.tensordot(power_spectrogram, self._mel_weight_matrix, axes=1)
        mel_spectrogram.set_shape(power_spectrogram.shape[:-1].concatenate([self.num_mel_bins]))

        log_mel = tf.math.log(mel_spectrogram + self.eps)

        # Normalize per sample untuk stabilitas training.
        mean = tf.reduce_mean(log_mel, axis=[1, 2], keepdims=True)
        std = tf.math.reduce_std(log_mel, axis=[1, 2], keepdims=True)
        log_mel = (log_mel - mean) / (std + self.eps)

        return tf.expand_dims(log_mel, axis=-1)  # (batch, time, mel, channel)

    def get_config(self):
        config = super().get_config()
        config.update({
            "sample_rate": self.sample_rate,
            "frame_length": self.frame_length,
            "frame_step": self.frame_step,
            "fft_length": self.fft_length,
            "num_mel_bins": self.num_mel_bins,
            "lower_edge_hertz": self.lower_edge_hertz,
            "upper_edge_hertz": self.upper_edge_hertz,
            "eps": self.eps,
        })
        return config

## 7. Build model Functional API: Raw Audio → MelSpectrogramLayer → CNN 2D → Softmax 20

In [8]:
def conv_block(x, filters: int, block_id: int, dropout_rate: float):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"cnn_block_{block_id}_conv2d_1")(x)
    x = tf.keras.layers.BatchNormalization(name=f"cnn_block_{block_id}_bn_1")(x)
    x = tf.keras.layers.Activation("relu", name=f"cnn_block_{block_id}_relu_1")(x)

    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"cnn_block_{block_id}_conv2d_2")(x)
    x = tf.keras.layers.BatchNormalization(name=f"cnn_block_{block_id}_bn_2")(x)
    x = tf.keras.layers.Activation("relu", name=f"cnn_block_{block_id}_relu_2")(x)

    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name=f"cnn_block_{block_id}_maxpool")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name=f"cnn_block_{block_id}_dropout")(x)
    return x


def build_heartz_model(config: HeartzConfig) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(config.num_samples,), name="raw_audio_waveform_16khz_1s")

    x = MelSpectrogramLayer(
        sample_rate=config.sample_rate,
        frame_length=config.frame_length,
        frame_step=config.frame_step,
        fft_length=config.fft_length,
        num_mel_bins=config.num_mel_bins,
        lower_edge_hertz=config.lower_edge_hertz,
        upper_edge_hertz=config.upper_edge_hertz,
        name="mel_spectrogram_layer",
    )(inputs)

    x = conv_block(x, filters=32, block_id=1, dropout_rate=0.10)
    x = conv_block(x, filters=64, block_id=2, dropout_rate=0.15)
    x = conv_block(x, filters=128, block_id=3, dropout_rate=0.20)

    x = tf.keras.layers.Conv2D(192, 3, padding="same", use_bias=False, name="final_conv2d")(x)
    x = tf.keras.layers.BatchNormalization(name="final_bn")(x)
    x = tf.keras.layers.Activation("relu", name="final_relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_2d")(x)

    x = tf.keras.layers.Dense(128, activation="relu", name="dense_projection")(x)
    x = tf.keras.layers.Dropout(0.35, name="classifier_dropout")(x)
    outputs = tf.keras.layers.Dense(config.num_classes, activation="softmax", name="syllable_softmax_20_classes")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name="heartz_raw_audio_cnn2d")

model = build_heartz_model(CONFIG)
model.summary()

Model: "heartz_raw_audio_cnn2d"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ raw_audio_waveform_16khz_1s     │ (None, 16000)          │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mel_spectrogram_layer           │ (None, 100, 64, 1)     │             0 │
│ (MelSpectrogramLayer)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_conv2d_1 (Conv2D)   │ (None, 100, 64, 32)    │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_bn_1                │ (None, 100, 64, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_relu_1 (Activation) │ (None, 100, 64, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_conv2d_2 (Conv2D)   │ (None, 100, 64, 32)    │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_bn_2                │ (None, 100, 64, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_relu_2 (Activation) │ (None, 100, 64, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_maxpool             │ (None, 50, 32, 32)     │             0 │
│ (MaxPooling2D)                  │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_1_dropout (Dropout)   │ (None, 50, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_conv2d_1 (Conv2D)   │ (None, 50, 32, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_bn_1                │ (None, 50, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_relu_1 (Activation) │ (None, 50, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_conv2d_2 (Conv2D)   │ (None, 50, 32, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_bn_2                │ (None, 50, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_relu_2 (Activation) │ (None, 50, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_maxpool             │ (None, 25, 16, 64)     │             0 │
│ (MaxPooling2D)                  │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_2_dropout (Dropout)   │ (None, 25, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_3_conv2d_1 (Conv2D)   │ (None, 25, 16, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_block_3_bn_1                │ (None, 25, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 537,012 (2.05 MB)

 Trainable params: 535,732 (2.04 MB)

 Non-trainable params: 1,280 (5.00 KB)

# Tes Visualisasi spectogram

In [ ]:
# Visualisasi & inspect data: waveform -> log-mel spectrogram (output MelSpectrogramLayer)
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError(
        "Butuh matplotlib untuk visualisasi. Install dulu: pip install matplotlib"
    ) from exc

# Ambil 1 sample dari train_ds (waveform & label one-hot)
x_batch, y_batch = next(iter(train_ds.take(1)))
waveform = x_batch[0].numpy()  # (16000,) float32
label_idx = int(np.argmax(y_batch[0].numpy()))
label_name = CLASS_NAMES[label_idx]

# Ambil output layer mel yang ada di dalam model
mel_layer = model.get_layer("mel_spectrogram_layer")
log_mel = mel_layer(x_batch[:1], training=False).numpy()  # (1, time, mel, 1)
log_mel_2d = log_mel[0, :, :, 0]

print("Waveform:", waveform.shape, waveform.dtype, "min/max", float(waveform.min()), float(waveform.max()))
print("Label:", label_idx, label_name)
print("Log-mel spectrogram:", log_mel.shape, log_mel.dtype, "min/max", float(log_mel_2d.min()), float(log_mel_2d.max()))

# Plot
t = np.arange(waveform.shape[0]) / float(CONFIG.sample_rate)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(t, waveform)
axes[0].set_title(f"Waveform (label={label_name})")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

im = axes[1].imshow(log_mel_2d.T, aspect="auto", origin="lower")
axes[1].set_title("Log-Mel Spectrogram (normalized)")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Mel bin")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 8. Sanity check arsitektur

In [9]:
def sanity_check_model(model: tf.keras.Model):
    checks = []
    checks.append((isinstance(model, tf.keras.Model) and not isinstance(model, tf.keras.Sequential), "Keras Functional API model"))
    checks.append((tuple(model.input_shape[1:]) == (CONFIG.num_samples,), f"Raw audio input shape {model.input_shape}"))
    checks.append((model.layers[1].__class__.__name__ == "MelSpectrogramLayer", f"First layer after Input: {model.layers[1].__class__.__name__}"))
    checks.append((tuple(model.output_shape[1:]) == (CONFIG.num_classes,), f"Output shape {model.output_shape}"))

    forbidden = ["resnet", "vgg", "mobilenet", "efficientnet", "inception", "xception", "densenet"]
    layer_text = " ".join([layer.name.lower() + " " + layer.__class__.__name__.lower() for layer in model.layers])
    checks.append((not any(name in layer_text for name in forbidden), "No pretrained application models detected"))

    for passed, message in checks:
        print(("[PASS]" if passed else "[FAIL]"), message)
        if not passed:
            raise AssertionError(message)

sanity_check_model(model)

[PASS] Keras Functional API model
[PASS] Raw audio input shape (None, 16000)
[PASS] First layer after Input: MelSpectrogramLayer
[PASS] Output shape (None, 20)
[PASS] No pretrained application models detected


## 9. Custom loss + optimizer + metrics

In [10]:
def heartz_categorical_crossentropy(y_true, y_pred, label_smoothing: float = 0.1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    if label_smoothing > 0:
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true = y_true * (1.0 - label_smoothing) + (label_smoothing / num_classes)

    per_sample_loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(per_sample_loss)

optimizer = tf.keras.optimizers.Adam(learning_rate=CONFIG.learning_rate)

## 10. Custom training loop dengan tf.GradientTape

Tidak ada `model.fit()` di cell ini.

In [ ]:
train_writer = tf.summary.create_file_writer(str(LOG_DIR / "train"))
val_writer = tf.summary.create_file_writer(str(LOG_DIR / "val"))

@tf.function
def train_step(model, x_batch, y_batch):
    with tf.GradientTape() as tape:
        y_pred = model(x_batch, training=True)
        loss = heartz_categorical_crossentropy(
            y_batch,
            y_pred,
            label_smoothing=CONFIG.label_smoothing,
        )

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss, y_pred

@tf.function
def val_step(model, x_batch, y_batch):
    y_pred = model(x_batch, training=False)
    loss = heartz_categorical_crossentropy(
        y_batch,
        y_pred,
        label_smoothing=CONFIG.label_smoothing,
    )
    return loss, y_pred


def run_training(model, train_ds, val_ds=None):
    best_val_loss = float("inf")
    best_weights = None
    wait = 0
    history = []

    for epoch in range(1, CONFIG.epochs + 1):
        start = time.time()
        train_loss_metric = tf.keras.metrics.Mean()
        train_acc_metric = tf.keras.metrics.CategoricalAccuracy()

        for x_batch, y_batch in train_ds:
            loss, y_pred = train_step(model, x_batch, y_batch)
            train_loss_metric.update_state(loss)
            train_acc_metric.update_state(y_batch, y_pred)

        train_loss = float(train_loss_metric.result().numpy())
        train_acc = float(train_acc_metric.result().numpy())

        if val_ds is not None:
            val_loss_metric = tf.keras.metrics.Mean()
            val_acc_metric = tf.keras.metrics.CategoricalAccuracy()
            for x_batch, y_batch in val_ds:
                loss, y_pred = val_step(model, x_batch, y_batch)
                val_loss_metric.update_state(loss)
                val_acc_metric.update_state(y_batch, y_pred)
            val_loss = float(val_loss_metric.result().numpy())
            val_acc = float(val_acc_metric.result().numpy())
        else:
            val_loss = train_loss
            val_acc = train_acc

        with train_writer.as_default():
            tf.summary.scalar("loss", train_loss, step=epoch)
            tf.summary.scalar("accuracy", train_acc, step=epoch)
        with val_writer.as_default():
            tf.summary.scalar("loss", val_loss, step=epoch)
            tf.summary.scalar("accuracy", val_acc, step=epoch)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "seconds": time.time() - start,
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d}/{CONFIG.epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
            f"{row['seconds']:.1f}s"
        )

        # Manual Early Stopping
        if val_loss < best_val_loss - CONFIG.min_delta:
            best_val_loss = val_loss
            best_weights = model.get_weights()
            wait = 0
            print("  -> best val_loss improved, weights cached")
        else:
            wait += 1
            print(f"  -> no improvement, wait={wait}/{CONFIG.early_stopping_patience}")
            if wait >= CONFIG.early_stopping_patience:
                print("Early stopping triggered.")
                break

    if best_weights is not None:
        model.set_weights(best_weights)
        print("Best weights restored.")

    return history

history = run_training(model, train_ds, val_ds)

Epoch 001/50 | train_loss=0.8268 train_acc=0.9686 | val_loss=0.9075 val_acc=0.9433 | 212.5s
  -> best val_loss improved, weights cached
Epoch 002/50 | train_loss=0.7453 train_acc=0.9903 | val_loss=0.7343 val_acc=0.9833 | 195.0s
  -> best val_loss improved, weights cached
Epoch 003/50 | train_loss=0.7249 train_acc=0.9922 | val_loss=0.6655 val_acc=0.9967 | 334.0s
  -> best val_loss improved, weights cached
Epoch 004/50 | train_loss=0.7014 train_acc=0.9965 | val_loss=0.6393 val_acc=0.9983 | 509.7s
  -> best val_loss improved, weights cached


## 11. Evaluasi test set

In [ ]:
def evaluate_dataset(model, ds, split_name="test"):
    if ds is None:
        print(f"No {split_name} dataset available.")
        return None

    loss_metric = tf.keras.metrics.Mean()
    acc_metric = tf.keras.metrics.CategoricalAccuracy()

    y_true_all = []
    y_pred_all = []

    for x_batch, y_batch in ds:
        y_pred = model(x_batch, training=False)
        loss = heartz_categorical_crossentropy(y_batch, y_pred, label_smoothing=0.0)
        loss_metric.update_state(loss)
        acc_metric.update_state(y_batch, y_pred)
        y_true_all.extend(tf.argmax(y_batch, axis=1).numpy().tolist())
        y_pred_all.extend(tf.argmax(y_pred, axis=1).numpy().tolist())

    result = {
        f"{split_name}_loss": float(loss_metric.result().numpy()),
        f"{split_name}_accuracy": float(acc_metric.result().numpy()),
    }
    print(result)
    return result, y_true_all, y_pred_all

test_result = evaluate_dataset(model, test_ds, "test")

## 12. Export model dan metadata

In [ ]:
# Simpan label dan config untuk dipakai FastAPI.
with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, indent=2)

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(asdict(CONFIG), f, indent=2)

# Keras native format, paling praktis untuk load di FastAPI.
model.save(KERAS_MODEL_PATH)
print("Saved Keras model to:", KERAS_MODEL_PATH.resolve())

# SavedModel export untuk kebutuhan produksi/deployment TensorFlow.
if SAVED_MODEL_DIR.exists():
    shutil.rmtree(SAVED_MODEL_DIR)

try:
    model.export(str(SAVED_MODEL_DIR))  # Keras 3
except Exception as exc:
    print("model.export failed, fallback to tf.saved_model.save:", repr(exc))
    tf.saved_model.save(model, str(SAVED_MODEL_DIR))

print("SavedModel directory:", SAVED_MODEL_DIR.resolve())
print("Class names:", CLASS_NAMES_PATH.resolve())
print("Config:", CONFIG_PATH.resolve())

## 13. Quick inference test dari satu file `.wav`

In [ ]:
def load_one_wav_for_inference(path: str) -> tf.Tensor:
    audio_binary = tf.io.read_file(path)
    audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1, desired_samples=CONFIG.num_samples)
    tf.debugging.assert_equal(sample_rate, CONFIG.sample_rate, message="Sample rate harus 16000 Hz")
    waveform = tf.squeeze(audio, axis=-1)
    return tf.expand_dims(waveform, axis=0)  # (1, 16000)

# Ganti path ini ke file test yang lu punya.
SAMPLE_WAV = test_paths[0] if test_paths else paths[0]
waveform = load_one_wav_for_inference(SAMPLE_WAV)
probs = model(waveform, training=False).numpy()[0]
pred_idx = int(np.argmax(probs))
confidence = float(probs[pred_idx])

print("File:", SAMPLE_WAV)
print("Prediction:", CLASS_NAMES[pred_idx])
print("Confidence:", confidence)

## 14. TensorBoard

Jalankan di terminal dari folder `machine-learning`:

```bash
tensorboard --logdir outputs/logs
```

Lalu buka `http://localhost:6006`.

## 15. Setelah notebook selesai

File yang harus muncul:

```text
outputs/heartz_model.keras
outputs/saved_model/
outputs/class_names.json
outputs/heartz_config.json
outputs/logs/
```

Setelah itu baru jalankan API:

```bash
uvicorn api.main:app --reload
```

Swagger:

```text
http://127.0.0.1:8000/docs
```